# Quick-Start: Image Attribution with Captum

**Goal:** Explain any ImageNet CNN prediction in under 2 minutes.

**What you get:** IG heatmap + GradCAM overlay + Occlusion map for any image/model.

**Prerequisites:** PyTorch, torchvision, captum installed.

```bash
pip install captum torchvision
```

---

In [ ]:
import sys; sys.path.insert(0, '../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

apply_course_theme()
apply_plot_theme()

In [ ]:
# ============================================================
# STEP 1: Imports and Model Setup
# ============================================================
import torch
import torchvision.models as models
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
import urllib.request
import io
import json
from PIL import Image

from captum.attr import IntegratedGradients, LayerGradCam, Occlusion, LayerAttribution
import torch.nn.functional as F

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# --- CONFIGURE HERE: swap for any torchvision CNN ---
model = models.resnet50(weights='IMAGENET1K_V1').to(DEVICE)
model.eval()

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

preprocess = transforms.Compose([
    transforms.Resize(256), transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

def denormalize(t):
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std  = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    return torch.clamp(t.cpu() * std + mean, 0, 1)

print("Model ready.")

In [ ]:
# ============================================================
# STEP 2: Load Your Image
# ============================================================

# --- CONFIGURE HERE: swap for your image URL or local path ---
IMAGE_SOURCE = "https://upload.wikimedia.org/wikipedia/commons/thumb/2/26/YellowLabradorLooking_new.jpg/320px-YellowLabradorLooking_new.jpg"
# Or load from file:
# IMAGE_SOURCE = "/path/to/your/image.jpg"

# Load image
if IMAGE_SOURCE.startswith('http'):
    try:
        with urllib.request.urlopen(IMAGE_SOURCE, timeout=10) as resp:
            raw_image = Image.open(io.BytesIO(resp.read())).convert('RGB')
    except Exception:
        raw_image = Image.fromarray(np.random.randint(80, 200, (320, 320, 3), dtype=np.uint8))
else:
    raw_image = Image.open(IMAGE_SOURCE).convert('RGB')

# Preprocess
input_tensor = preprocess(raw_image).unsqueeze(0).to(DEVICE)
img_np = denormalize(input_tensor.squeeze(0)).permute(1, 2, 0).numpy()
baseline = torch.zeros_like(input_tensor)

# Get prediction
try:
    LABELS_URL = "https://raw.githubusercontent.com/anishathalye/imagenet-simple-labels/master/imagenet-simple-labels.json"
    with urllib.request.urlopen(LABELS_URL, timeout=5) as resp:
        labels = json.loads(resp.read().decode())
except Exception:
    labels = [f"class_{i}" for i in range(1000)]

with torch.no_grad():
    logits = model(input_tensor)
    probs  = torch.softmax(logits, dim=1)
    top5   = probs.topk(5, dim=1)

top_class = top5.indices[0, 0].item()
print(f"Top-5 Predictions:")
for i in range(5):
    cls = top5.indices[0, i].item()
    p   = top5.values[0, i].item()
    print(f"  {i+1}. {labels[cls]:35s} {p:.1%}")

In [ ]:
# ============================================================
# STEP 3: Run All Three Attribution Methods
# ============================================================

# --- Integrated Gradients ---
ig = IntegratedGradients(model)
inp = input_tensor.clone().requires_grad_(True)
ig_attr, ig_delta = ig.attribute(
    inp, baselines=baseline, target=top_class,
    n_steps=50, return_convergence_delta=True
)
ig_map = ig_attr.abs().mean(1).squeeze(0).detach().cpu().numpy()
ig_map = np.clip((ig_map - np.percentile(ig_map, 1)) /
                  (np.percentile(ig_map, 99) - np.percentile(ig_map, 1) + 1e-8), 0, 1)
print(f"IG  convergence delta: {ig_delta.item():.5f}")

# --- GradCAM (layer4) ---
# --- CONFIGURE HERE: swap for the target layer in your model ---
target_layer = model.layer4[-1]  # For ResNet-50
# For VGG-16: target_layer = model.features[-3]

lg = LayerGradCam(model, target_layer)
gc_attr = lg.attribute(input_tensor, target=top_class)
gc_attr_up = LayerAttribution.interpolate(gc_attr, (224, 224), 'bilinear')
gc_map = torch.relu(gc_attr_up.sum(1)).squeeze(0).detach().cpu()
gc_map = ((gc_map - gc_map.min()) / (gc_map.max() - gc_map.min() + 1e-8)).numpy()
print("GradCAM: done")

# --- Occlusion ---
occ = Occlusion(model)
occ_attr = occ.attribute(
    input_tensor, strides=(3, 8, 8), target=top_class,
    sliding_window_shapes=(3, 15, 15), baselines=0
)
occ_map = occ_attr.abs().mean(1).squeeze(0).detach().cpu().numpy()
occ_map = np.clip((occ_map - np.percentile(occ_map, 1)) /
                   (np.percentile(occ_map, 99) - np.percentile(occ_map, 1) + 1e-8), 0, 1)
print("Occlusion: done")

In [ ]:
# ============================================================
# STEP 4: Visualize Results
# ============================================================

pred_label = labels[top_class]
pred_prob  = top5.values[0, 0].item()

fig, axes = plt.subplots(2, 4, figsize=(22, 10))
fig.suptitle(
    f'Attribution Methods — Prediction: "{pred_label}" ({pred_prob:.1%})\n'
    f'IG delta={ig_delta.item():.4f}',
    fontsize=13
)

panels = [
    ('Input Image',           img_np,  None),
    ('Integrated Gradients',  ig_map,  'hot'),
    ('GradCAM (layer4)',      gc_map,  'jet'),
    ('Occlusion (15×15, s=8)', occ_map, 'hot'),
]

for col, (title, hmap, cmap) in enumerate(panels):
    # Row 0: pure heatmap / image
    if cmap:
        im = axes[0, col].imshow(hmap, cmap=cmap, vmin=0, vmax=1)
        plt.colorbar(im, ax=axes[0, col], fraction=0.046)
    else:
        axes[0, col].imshow(hmap)
    axes[0, col].set_title(title, fontsize=10)
    axes[0, col].axis('off')

    # Row 1: overlay on image
    axes[1, col].imshow(img_np)
    if cmap:
        axes[1, col].imshow(hmap, alpha=0.55, cmap=cmap,
                             vmin=np.percentile(hmap, 60))
    axes[1, col].set_title(f'{title} overlay', fontsize=9)
    axes[1, col].axis('off')

plt.tight_layout()
plt.savefig('attribution_results.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: attribution_results.png")

In [ ]:
# ============================================================
# STEP 5: Validation Summary
# ============================================================

from scipy.stats import spearmanr

rho_ig_gc,  _ = spearmanr(ig_map.flatten(), gc_map.flatten())
rho_ig_occ, _ = spearmanr(ig_map.flatten(), occ_map.flatten())
rho_gc_occ, _ = spearmanr(gc_map.flatten(), occ_map.flatten())

print("Attribution Quality Summary")
print("=" * 40)
print(f"Prediction:            {pred_label} ({pred_prob:.1%})")
print(f"IG convergence delta:  {ig_delta.item():.5f}",
      "OK" if abs(ig_delta.item()) < 0.05 else "WARNING: increase n_steps")
print()
print("Method Agreement (Spearman ρ):")
print(f"  IG vs GradCAM:     ρ = {rho_ig_gc:.3f}")
print(f"  IG vs Occlusion:   ρ = {rho_ig_occ:.3f}")
print(f"  GradCAM vs Occ:    ρ = {rho_gc_occ:.3f}")
print()
print("Interpretation:")
mean_rho = np.mean([rho_ig_gc, rho_ig_occ, rho_gc_occ])
if mean_rho > 0.5:
    print(f"  High agreement (mean ρ={mean_rho:.2f}): attribution is robust.")
elif mean_rho > 0.3:
    print(f"  Moderate agreement (mean ρ={mean_rho:.2f}): generally consistent.")
else:
    print(f"  Low agreement (mean ρ={mean_rho:.2f}): investigate model behavior.")